In [39]:
import json
import requests
import pandas
import asyncio
import regex as re

from dotenv import load_dotenv
import os
load_dotenv()
PUBMED_KEY = os.getenv("PUBMED_KEY")
MY_EMAIL = os.getenv("MY_EMAIL")

# Fetching abstract

In [40]:
with open('training14b.json', 'r') as f:
    training_data = json.load(f)

    questions = training_data['questions']
    setID = set()
    for i in range(len(questions)):
        links = questions[i]['documents']
        ids = [re.findall(r"pubmed\/(\d+)", link) for link in links]
        ids = [item for sublist in ids for item in sublist]
        setID.update(ids)
    print(len(setID))

57996


In [41]:
import requests
from bs4 import BeautifulSoup
import time

BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
BATCH_SIZE = 200
RATE_LIMIT_DELAY = 0.34  # ~3 req/s; lower to 0.11 with an API key

def chunked(iterable, size):
    """Yield successive chunks of `size` from `iterable`."""
    iterable = list(iterable)
    for i in range(0, len(iterable), size):
        yield iterable[i:i + size]

def fetch_batch(id_batch, api_key=None, email=None, tool=None):
    """Fetch abstracts for a batch of PubMed IDs. Returns dict {pmid: [abstract_parts]}."""
    params = {
        "db": "pubmed",
        "id": ",".join(id_batch),
        "rettype": "abstract",
        "retmode": "xml",
    }
    if api_key:
        params["api_key"] = api_key
    if email:
        params["email"] = email
    if tool:
        params["tool"] = tool
    
    # POST is safer than GET for large ID lists
    response = requests.post(BASE_URL, data=params)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, "lxml-xml")
    results = {}
    for article in soup.select("PubmedArticle"):
        pmid_tag = article.select_one("PMID")
        if pmid_tag is None:
            continue
        pmid = pmid_tag.get_text()
        abstract_parts = [t.get_text() for t in article.select("AbstractText")]
        results[pmid] = abstract_parts
    return results

def get_abstracts(id_list, api_key=None, email=None, tool=None):
    all_results = {}
    for i, batch in enumerate(chunked(id_list, BATCH_SIZE)):
        batch_start = i * BATCH_SIZE
        batch_end = min((i + 1) * BATCH_SIZE, len(id_list))
        print(f"Processing batch {i+1}: IDs {batch_start}-{batch_end} of {len(id_list)}")
        all_results.update(fetch_batch(batch, api_key, email, tool))
        print(f"Completed batch {i+1}: {len(all_results)} abstracts retrieved so far")
        time.sleep(RATE_LIMIT_DELAY)
    return all_results

The script proposed above works just fine for fetching the abstracts mentioned above, and only fails on a very small subset of ids mentioned a little lower due to problems in 1) the format of the paper, 2) the fact that the paper was still under review or 3) because the paper was a duplicate of another. 

The script had to be retried multiple time not because the script itself was failing in any notable way, but because the API it was sending pings to was sometimes failing under the load. Since fetching the abstract takes about 15 minutes total with batches of 200 and a delay of 0.34 seconds, and that it only needed to go fully once, it was decided to just retry it a few more time rather than to implement more complicated save point and retry mechanismes. 

In [42]:
from IPython.display import clear_output
# This is not elegant, but it works. 
# There were problem where I would get a 502 error randomly
# I should have implemented a more robust retry system, or one that writes to the file as it goes and retry the ones that fails, oooooooooor
# I could just retry a few bunch of time and hope it passes.  
# while True:
#     try:
#         abstract_list = get_abstracts(setID, api_key=PUBMED_KEY, email=MY_EMAIL)
#         break
#     except Exception as e:
#         print(f"Retrying after exception: {e}")
#         clear_output(wait=True)
#         time.sleep(1)
# with open('abstract_list.json', 'w', encoding='utf-8') as out:
#     json.dump(abstract_list, out, ensure_ascii=False, indent=2)
# print(f"Saved {len(abstract_list)} abstracts to abstract_list.json")


In [43]:
with open('abstract_list.json', 'r', encoding='utf-8') as f:
    abstract_data = json.load(f)

missing_ids = setID - set(abstract_data.keys())
print(f"Number of missing abstracts: {len(missing_ids)}")
print("Missing IDs:")
for pmid in sorted(missing_ids):
    print(pmid)

Number of missing abstracts: 65
Missing IDs:
20007090
20301293
20301308
20301331
20301416
20301420
20301427
20301454
20301462
20301466
20301494
20301510
20301577
20301585
20301588
20301628
20301779
20639591
20641567
21249951
21250223
21284896
21413253
21473027
21473029
21952424
22129433
22550943
22787616
22787626
22855961
23104528
23304742
23658991
23689657
23833797
23890950
23986914
24144986
24212220
24305278
24305403
24960551
25064957
25383801
25401082
26989023
27399411
27399455
27742610
27924029
27940438
28976850
29653952
29676625
29934319
30242068
32859585
33097476
33409349
34468351
34766984
34788240
36459075
37800882


In [44]:
missing_abstract = get_abstracts(missing_ids, api_key=PUBMED_KEY, email=MY_EMAIL)

Processing batch 1: IDs 0-65 of 65
Completed batch 1: 1 abstracts retrieved so far


# Sentence tokenizing the abstract

In [59]:
import scispacy
import spacy

nlp = spacy.load("en_core_sci_md")
with open('abstract_list.json', 'r', encoding='utf-8') as f:
    abstracts = json.load(f)

In [ ]:
import pprint

abstract_data_tokenized = {}
total_len = len(abstracts)
for i, pmid in enumerate(abstracts):
    abstract_list = abstracts[pmid]
    if len(abstract_list) > 1:
        abstract = " ".join(abstract_list)
    else:
        try:
            abstract = abstract_list[0]
        except:
            pass
    doc = nlp(abstract)
    sentences = list(doc.sents)

    abstract_data_tokenized[pmid] = sentences
    print(f"Successfully tokenized abstract {i}/{total_len}")


In [70]:
with open('abstract_list_tokenized.json', 'w', encoding='utf-8') as f:
    json.dump(abstract_data_tokenized, f, ensure_ascii=False, indent=2)

TypeError: Object of type Span is not JSON serializable

In [71]:
import json
import spacy
from tqdm import tqdm

nlp = spacy.load(
    "en_core_sci_md",
    disable=["tagger", "ner", "lemmatizer", "attribute_ruler"]
)

with open("abstract_list.json", "r", encoding="utf-8") as f:
    abstracts = json.load(f)

pmids = []
texts = []
for pmid, abstract_list in abstracts.items():
    if not abstract_list:
        continue
    pmids.append(pmid)
    texts.append(" ".join(abstract_list))

abstract_data_tokenized = {}

for pmid, doc in tqdm(
    zip(pmids, nlp.pipe(texts, batch_size=50, n_process=1)),
    total=len(pmids),
    desc="Tokenizing abstracts",
):
    abstract_data_tokenized[pmid] = [sent.text for sent in doc.sents]

Tokenizing abstracts: 100%|██████████| 57889/57889 [26:09<00:00, 36.87it/s]


In [72]:
with open('abstract_list_tokenized.json', 'w', encoding='utf-8') as f:
    json.dump(abstract_data_tokenized, f, ensure_ascii=False, indent=2)